# Edit N-grams
Demo code for training an $N$-gram LM on string edits.
Uses `Levenshtein` package to convert a (query, target) pair of strings to a list of (intab, outtab) pairs.

In [1]:
from Levenshtein import editops

`Levenshtein.editops` returns a list of triples `('edit', i, j)` 

In [2]:
editops("foo", "boo")

[('replace', 0, 0)]

Convert the list of triples to a list of character pairs.

In [3]:
_epsilon = "<Empty>"


def get_edit_corpus(corpus: list[tuple[str, str]]) -> list[tuple[str, str]]:
    edit_pairs = []
    for in_word, out_word in corpus:
        edit_pairs.extend(get_edit_pairs(in_word, out_word))
    return edit_pairs


def get_edit_pairs(in_word: str, out_word: str) -> list[tuple[str, str]]:
    operations = editops(in_word, out_word)
    char_pairs = [
        operation_to_char_pair(operation, in_word, out_word) for operation in operations
    ]
    return char_pairs


def operation_to_char_pair(
    operation: tuple[str, int, int], in_word: str, out_word: str
) -> tuple[str, str]:
    operation_type, i, j = operation
    match operation_type:
        case "insert":
            return (_epsilon, out_word[j])
        case "replace":
            return (in_word[i], out_word[j])
        case "delete":
            return (in_word[i], _epsilon)


get_edit_pairs("food", "bfoobar")

[('<Empty>', 'b'), ('<Empty>', 'b'), ('<Empty>', 'a'), ('d', 'r')]

## Bigram LM estimation
Learn a bigram model to estimate $\mathrm{P}(c_j|c_i)$.

In [4]:
from nltk.lm import MLE
from string import ascii_lowercase

In [5]:
lm = MLE(2)

vocab = [_epsilon] + list(ascii_lowercase)
vocab[:5]

['<Empty>', 'a', 'b', 'c', 'd']

In [6]:
corpus = [
    ("foo", "fo"),
    ("foo", "vaa"),
    ("foo", "vaa"),
    ("foo", "vaa"),
    ("foo", "vaa"),
    ("foo", "vaa"),
    ("bar", "bat"),
    ("baz", "pid"),
]

edit_corpus = get_edit_corpus(corpus)
edit_corpus[:5]

[('o', '<Empty>'), ('f', 'v'), ('o', 'a'), ('o', 'a'), ('f', 'v')]

In [7]:
lm.fit([edit_corpus], vocabulary_text=vocab)

In [8]:
lm.entropy([('o', 'a')])

0.13750352374993496

In [9]:
lm.score(context='o', word='a')

0

Don't need NLTK though.
We can just use a $|V\cup\emptyset|\times|V\cup\emptyset|$ transition matrix (as long as we're sticking to bigrams).

In [ ]:
import numpy as np
from scipy.special import softmax
from scipy.stats import entropy, dirichlet

In [46]:
def get_random_transition_matrix(n, alpha=1.0):
    mat = np.random.dirichlet([alpha] * n, size=n)
    # probs = softmax(mat, axis=1)
    probs = mat / mat.sum(axis=1, keepdims=True)  # Ensure rows sum to 1

    return probs

mat = get_random_transition_matrix(len(vocab))
mat[0].sum(), mat.shape, mat.min(), mat.max()

(np.float64(1.0),
 (27, 27),
 np.float64(9.52812256530337e-06),
 np.float64(0.23393381550175724))

In [56]:
def kl_divergence_from_uniform(probs):
    uniform_dist = np.ones_like(probs) / len(probs)
    uniform_dist = uniform_dist / uniform_dist.sum()  # Ensure it's a valid distribution
    kl_div = entropy(probs, uniform_dist)
    return kl_div

for alpha in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    mat = get_random_transition_matrix(len(vocab), alpha=alpha)
    kl_div = np.mean([kl_divergence_from_uniform(row) for row in mat])
    print(f"Alpha: {alpha}, KL Divergence from Uniform: {kl_div:.4f}")

Alpha: 0.001, KL Divergence from Uniform: 3.2922
Alpha: 0.01, KL Divergence from Uniform: 2.8823
Alpha: 0.1, KL Divergence from Uniform: 1.7540
Alpha: 1.0, KL Divergence from Uniform: 0.3893
Alpha: 10.0, KL Divergence from Uniform: 0.0421
Alpha: 100.0, KL Divergence from Uniform: 0.0048
